<a href="https://colab.research.google.com/github/VinishReddyK/zipcast-epub/blob/main/notebooks/colab_zipcast_qwen3tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# zipcast: epub → audiobook (Qwen3-TTS)

This notebook does the GPU-heavy half of the pipeline. The other half runs on your
laptop: `zipcast book.epub -o book.zip` parses the epub and packs clean chapter
text + metadata + cover art into a zip.

**Steps to use this notebook:**
1. Run `zipcast` locally on your epub to produce `book.zip`.
2. Open this notebook in Colab, set **Runtime > Change runtime type > T4 GPU**.
3. Drag `book.zip` into the Colab **Files** panel (left sidebar) so it lands in `/content`.
4. Run every cell top to bottom. The final cell downloads the finished `.m4b`.


In [ ]:
import os

# reduce CUDA memory fragmentation on long multi-chapter runs (must be set
# before torch initializes a CUDA context)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# fill this in once you've pushed this project to GitHub
REPO_URL = "https://github.com/VinishReddyK/zipcast-epub.git"  # @param {type:"string"}
REPO_DIR = "/content/zipcast"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# install ffmpeg (for m4b muxing) and the Qwen3-TTS python dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q -r {REPO_DIR}/colab/requirements.txt


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. go to Runtime > Change runtime type > T4 GPU, "
          "otherwise synthesis will be very slow.")


## Locate and validate the zip

Looks in `/content` for the zip you uploaded, confirms it's a non-corrupt zip,
and confirms it's actually a `zipcast` bundle (has `metadata.json` + chapter `.txt` files)
before doing any GPU work.


In [ ]:
from pathlib import Path
from colab.generate import find_zip, validate_zip, extract_zip

zip_path = find_zip("/content")
validate_zip(zip_path)
print(f"validated: {zip_path.name}")

metadata = extract_zip(zip_path, Path("/content/zipcast_work/extract"))
print(f"\nbook: {metadata['title']}")
print(f"author: {metadata['author']}")
print(f"chapters: {len(metadata['chapters'])}")
for c in metadata["chapters"][:5]:
    print(f"  {c['index']:02d}. {c['title']}")
if len(metadata["chapters"]) > 5:
    print(f"  ... and {len(metadata['chapters']) - 5} more")


## Synthesize and build the m4b

Loads Qwen3-TTS, synthesizes each chapter (resumable -- re-running this cell skips
chapters whose `.wav` already exists in `/content/zipcast_work/wav`), then muxes
everything into a single `.m4b` with chapter markers and cover art via ffmpeg.


In [ ]:
SPEAKER = "ryan"  # @param ["ryan", "vivian", "sunny", "aria", "bella", "nova", "echo", "finn", "atlas"]
MODEL_NAME = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"  # @param {type:"string"}
BATCH_SIZE = 4  # @param {type:"integer"}
# chunks synthesized per model call. lower this (e.g. to 1-2) if you still hit
# CUDA out-of-memory errors on unusually long chapters.

from colab.generate import run_pipeline

output_path = run_pipeline(
    content_dir="/content",
    speaker=SPEAKER.lower(),
    model_name=MODEL_NAME,
    device=device,
    batch_size=BATCH_SIZE,
)
print(f"\naudiobook ready: {output_path}")


In [ ]:
from google.colab import files

files.download(str(output_path))
